# PIV demo: blind prediction on the Elman RNN case study

Re-runs the **Predict** stage end-to-end on a single CPU in under 4 minutes:
train a fresh blind cohort from raw seeds, apply the pre-registered
certificate (`certificate-freeze` git tag), and report operating
characteristics with Wilson 95% CIs.

The notebook cohort is 30 models / 2,000 epochs to stay inside the time
budget; the paper-scale population (300 models, 10,000 epochs) is
`make all` (GPU recommended). Numbers here are produced live by your run.

In [ ]:
# %pip install git+https://github.com/stef41/piv torch
import time
import piv
from piv.casestudy import CERTIFICATE, certify, persistence, train_population

print('pre-registered certificate:', piv.freeze(CERTIFICATE))

In [ ]:
t0 = time.time()
blind, _ = train_population(range(200, 230), k=3, H=8, epochs=2000,
                            device='cpu', log_every=500)
print(f'trained 30 blind models in {time.time()-t0:.0f}s')

In [ ]:
pers = persistence(blind)
cert = certify(blind)
y = pers >= 0.99
pred = cert['certified']
tp = int((pred & y).sum()); fp = int((pred & ~y).sum())
fn = int((~pred & y).sum()); tn = int((~pred & ~y).sum())
print(f'prevalence {int(y.sum())}/{len(y)}')
print(f'certified {int(pred.sum())}: tp={tp} fp={fp} fn={fn} tn={tn}')
if tp + fp: print('precision', tp/(tp+fp), piv.wilson_ci(tp, tp+fp))
if tp + fn: print('recall   ', tp/(tp+fn), piv.wilson_ci(tp, tp+fn))
print(f'total wall time {time.time()-t0:.0f}s')

Next stages: `scripts/surgery.py` (Intervene, matched controls +
cluster-bootstrap null) and `scripts/run_all.py` (full populations
including the k=12–25 hyperparameter-frozen OOD sweep).